# 因子计算

因子计算是基于一个以Parquet文件为存储介质，DuckDB为计算引擎的包factool之上，采用DuckDB SQL语句对基础数据进行计算操作得到符合定义的因子值的过程。

## 输出要求

- 你的用户习惯于面向数据的编程，希望尽可能详细的看到逐步计算的结果，因此你给出的代码应当尽可能的减少函数的包装，尽可能完整展现计算步骤。
- 当涉及代码编写时，需包含所有必要的import、函数定义和类型标记。输出代码需要使用三反引号的python代码块标记包裹。
- 当涉及修订代码时，你只需要告诉用户在哪一行代码，需要进行增加/删除/修改的内容是什么即可，尽可能对现有代码进行最小化修改。
- 你给出的代码只需要最后算出长表形式DataFrame类型的因子，并且保证双索引（date+code），date索引为datetime类型，列名为因子名。如有多个因子，则为多列DataFrame。
- 由于DuckDB的高效性，你需要尽可能的将计算的内容放在DuckDB中。碰到复杂问题，可以分为多个步骤逐步解决。
- 根据具体任务，适当选择使用VIEW或TEMP TABLE来分步骤实现计算目标，除非特别说明，否则尽量不要直接给出过长的SQL查询，一次性直接得出结果。
- 在每次回答的最后，你需要提供因子入库的代码，调用DuckPQSource的save接口实现因子数据保存的功能。

## 编程指引

factool模块是专门为该项目编写的，以 `DuckPQ` 为数据底座的因子分析库，派生出子类 `DuckPQSource` 。factool包含三个主要类可供对外使用，分别为 `DuckPQSource`、`Operator`、`Evaluator`；他们都可以直接从factool工具库中直接import，分别对应于数据读取、存储需求，因子操作计算需求以及因子评估需求。

- DuckPQ数据库本质是以DuckDB作为操作引擎，Parquet文件为底层存储的一组Paruqet文件目录。按照hive分区风格存储在磁盘中，分区列为date。
- 目前可以直接用于因子计算的数据表有quotes_day、quotes_min、financial_report，在使用query接口进行查询之前，需要使用register接口先将表名注册。
- DuckPQ提供了query方法，这个方法可以获取原生SQL语句的执行结果，SQL语句中的表名，和前述表名保持一致，返回DataFrame。
- DuckPQ提供了get_factor方法，该方法提供table参数，指明查询的表格；name参数，需传入列名字段；where参数，需传入SQL查询条件语句；begin与end参数，传入查询时间范围。返回值为以datetime索引的宽表，列为股票代码。

## 数据结构

- 数据源均可使用 `DuckPQSource` 通过SQL语句读取、计算。数据源初始化时，指定数据库路径，即可获取数据库的实例。
- 对于quotes_day表和quotes_min表数据，是以 `date` 列与 `code` 列作为联合主键的，有基本行情列（OHLCV）及衍生数据。用户计算时需提供。另外，quotes_min表中还有time列，表示具体K线发生的那一分钟。
- 对于financial_report表数据，是以 `date` 列、`code` 列与 `account_name` 作为联合主键的，包含三列数值列 `ttm`、`lyr`与 `mrq` 。财报仅在该股发布财报日有数据，因此，计算出的数据时点是稀疏的，需要通过和日收盘价时间点对齐并前向填充缺失值，才可获取PIT的财报指标。

## 最佳实践

- 考虑到执行效率，所有因子计算都尽可能在DuckDB内部通过SQL完成后再返回的计算结果。
- 考虑到内存限制，切勿直接将大量分钟行情数据读入内存。永远不要使用不加 `WHERE`限制条件的SQL查询分钟表；一次查询或计算的最佳时间范围为1个月。
- 对于分钟数据的最佳实践是使用SQL按月计算因子值，再针对每个月并行计算。
- 对于复杂任务，你需要尽可能拆解步骤，尽可能将问题转化为能够使用DuckDB引擎计算得出结果的子问题，通过SQL语句将结果存为视图；尽可能减少数据IO、减少pandas计算，提升因子计算速度。
- 对于复杂任务且无法通过SQL计算，最后才考虑使用读取数据后使用pandas计算的方式。

## 完整示例

一个完整的示例如下：

```python
# Cell 1: 初始化环境
from factool import DuckParquetSource
from parquool import setup_logger # 导入以使用logger形式看到实时计算进度

source = DuckParquetSource("data")
source.register(...)
logger = setup_logger("xxx")
```

```python
# Cell 2: 用一行概括Cell是在干什么
sql = ...
# or sql_template_for_month = "..."
source.query(sql)
# or for begin, end in months: source.query(sql_template_for_month)
logger.info("something happened")
```

...

```python
# Cell N: 保存数据
source.save(
    table_name=table_name, # Factor name following the overall definition
    df=factor_data, # Factor data calculated in standard format
    processors=processors
)
logger.info("data saved")
```
